In [1]:
import boto3
from botocore.config import Config
from io import StringIO
import pandas as pd
import torch

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()
read_access_key = os.getenv("AWS_ACCESS_KEY_ID")
read_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket_name = os.getenv("AWS_BUCKET_NAME")
endpoint_url = os.getenv("AWS_ENDPOINT_URL")
file_key = "dar_tables/full_dataset_sequences.txt" 
local_file_path = 'dataset'

# Initialize S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=read_access_key,
    aws_secret_access_key=read_secret_key,
    endpoint_url=endpoint_url,
    config=Config(signature_version="s3v4"),
)


def fetch_dataset_from_s3(bucket_name, file_key):
    """
    Fetch dataset from a text file in an Amazon S3 bucket.
    :param bucket_name: Name of the S3 bucket.
    :param file_key: Path to the file in the bucket.
    :return: List of sequences and features.
    """
    print(f"Fetching file from S3: {file_key}")
    obj = s3.get_object(Bucket=bucket_name, Key=file_key)
    file_content = obj["Body"].read().decode("utf-8")
    
    # Assuming the file is tab-delimited with a column named 'sequence'
    df = pd.read_csv(StringIO(file_content), sep="\t")

    print(f"Loaded {len(df)} sequences from the S3 file.")
    return df

def save_df_to_s3(df, bucket_name, s3_folder, file_name):
    """
    Save a DataFrame to a specified folder in the S3 bucket as a CSV file.
    :param df: DataFrame to save.
    :param bucket_name: Name of the S3 bucket.
    :param s3_folder: Folder path within the bucket.
    :param file_name: Name of the file to save in the bucket.
    """
    s3_key = f"{s3_folder}/{file_name}"
    try:
        # Convert DataFrame to CSV in memory
        csv_buffer = StringIO()
        df.to_csv(csv_buffer, index=False)
        
        # Upload CSV to S3
        s3.put_object(Bucket=bucket_name, Key=s3_key, Body=csv_buffer.getvalue())
        print(f"File successfully uploaded to S3: {bucket_name}/{s3_key}")
    except Exception as e:
        print(f"Error uploading to S3: {e}")


In [3]:
import os
import boto3
import torch
import dask
import dask.dataframe as dd
from dask import delayed



bucket_name = "metabolic-atac-peaks"
folder_name = "embeddings/"

# List all .pt files in the embeddings folder
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=folder_name)
pt_files = [content["Key"] for content in response.get("Contents", []) if content["Key"].endswith(".pt")]

print(f"Found {len(pt_files)} embedding files.")

Found 40 embedding files.


In [5]:
@delayed
def load_pt_file_from_s3(s3_client, bucket_name, file_key):
    # Download the .pt file to a temporary location
    local_file = f"/tmp/{os.path.basename(file_key)}"
    s3_client.download_file(bucket_name, file_key, local_file)
    
    # Load the .pt file with PyTorch
    embeddings = torch.load(local_file)
    
    # Delete the temporary file after loading
    os.remove(local_file)
    
    return embeddings

# Create delayed tasks for each file
embedding_tasks = [load_pt_file_from_s3(s3, bucket_name, file_key) for file_key in pt_files]

In [7]:
import dask.array as da

# Compute the shape of a single embedding
example_embedding = embedding_tasks[0].compute()
embedding_shape = example_embedding.shape

# Stack all embeddings into a Dask array
dask_embeddings = da.stack([da.from_delayed(task, shape=embedding_shape, dtype=example_embedding.dtype) for task in embedding_tasks])

print(f"Shape of combined embeddings: {dask_embeddings.shape}")

TypeError: 'int' object is not subscriptable

In [9]:
import os
import torch
from dask.delayed import delayed
import dask.array as da

# Specify the paths to the two embedding files in S3
embedding_files = ["embeddings/embeddings_chunk_1.pt", "embeddings/embeddings_chunk_2.pt"]

# Function to load embeddings directly from S3 using PyTorch
@delayed
def load_embedding_from_s3(file_key):
    local_file = f"/tmp/{os.path.basename(file_key)}"
    s3.download_file(bucket_name, file_key, local_file)  # Download from S3
    embedding = torch.load(local_file)  # Load with PyTorch
    os.remove(local_file)  # Cleanup temporary file
    return embedding

# Create delayed tasks for the specified files
embedding_tasks = [load_embedding_from_s3(file) for file in embedding_files]

# Compute shape and dtype of the first embedding
example_embedding = embedding_tasks[0].compute()
embedding_shape = example_embedding.shape
embedding_dtype = example_embedding.numpy().dtype

# Stack embeddings into a Dask array
dask_embeddings = da.stack([
    da.from_delayed(task, shape=embedding_shape, dtype=embedding_dtype)
    for task in embedding_tasks
])

# Perform computations on Dask array
print(f"Shape of combined embeddings: {dask_embeddings.shape}")
mean_vector = dask_embeddings.mean(axis=0).compute()
print("Mean Vector Computed:", mean_vector)

: 